# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [7]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm

df = pd.read_csv('customers.csv')
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  40 non-null     int64  
 1   age          40 non-null     int64  
 2   income       38 non-null     float64
 3   spending     40 non-null     float64
 4   region       40 non-null     str    
 5   subscribed   40 non-null     str    
dtypes: float64(2), int64(2), str(2)
memory usage: 2.0 KB


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [8]:
# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
# HINT 2: one category label is written in the wrong case -> standardize it
# HINT 3: one value is impossible for its column -> treat it as missing, then fill
# tip: check .isna().sum() and the unique values of the category column

# Look first
print(df.isna().sum())
print(df["region"].unique())
print(df.describe())

# 1. Standardize region
df["region"] = df["region"].replace("east", "East")

# 2. Impossible income (-50) -> missing
df.loc[df["income"] <= 0, "income"] = np.nan

# 3. Fill missing income with the median
df["income"] = df["income"].fillna(df["income"].median())

# Check
print(df.isna().sum())
print(df["region"].unique())
print(df["income"].describe())

customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0
dtype: int64
<StringArray>
['West', 'South', 'East', 'east']
Length: 4, dtype: str
       customer_id        age      income   spending
count    40.000000  40.000000   38.000000  40.000000
mean     20.500000  40.925000   83.007895  54.520000
std      11.690452  11.474358   27.628563   9.377912
min       1.000000  21.000000  -50.000000  36.000000
25%      10.750000  32.500000   72.975000  48.750000
50%      20.500000  41.000000   84.700000  53.650000
75%      30.250000  51.000000   98.500000  62.650000
max      40.000000  59.000000  114.500000  74.700000
customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0
dtype: int64
<StringArray>
['West', 'South', 'East']
Length: 3, dtype: str
count     40.000000
mean      86.460000
std       16.080165
min       53.300000
25%       75.075000
50%       84.700000
75%       97.100000
max      114.500000
N

**Explain each issue:** what you found and how you handled it.

*Your answer:*

1. Missing values: Income had 2 missing values (customers 4 and 18). I filled them with the median, not the mean,because the median isn't pulled around by extreme values.

2.Impossible value: Customer 26 had an income of -50 and income can't be negative. I set it to NaN and then filled it with the median (84.7) same as the other missing values.

## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [13]:
# data types: income = ?, spending = ?
# tool: ?
# hint: two continuous variables moving together

# Q1: Correlation between income and spending

correlation, p_value = stats.pearsonr(df["income"], df["spending"])

print("Correlation:", correlation)
print("p-value:", p_value)

Correlation: 0.7172222861355748
p-value: 1.9185294399175956e-07


*Your answer:*

Yes. Income and spending have a strong positive correlation (r=0.72, p <0.001)
, so customers who earn more tend to spend more.

### Q2. Is a customer's region related to whether they subscribed?

In [9]:
# data types: region = ?, subscribed = ?
# tool: ?
# hint: two categorical variables -> build a table with pd.crosstab, then test it

ct = pd.crosstab(df["region"], df["subscribed"])
print(ct)
chi2, p, dof, expected = stats.chi2_contingency(ct)
print("chi2:", chi2, "p-value:", p, "dof:", dof)

subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1
chi2: 13.161904761904761 p-value: 0.0013865281628130974 dof: 2


*Your answer:*

Yes. Chi-square = 13.16, df = 2, p= 0.0014, so region is significantly related to subscribing. East customers subscribe the most, and West the least.


### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [11]:
# data types: predicting a continuous outcome from TWO predictors
# tool: ?
# hint: remember to add a constant; look at R-squared and each predictor's p-value

X = sm.add_constant(df[["income", "age"]])
y = df["spending"]
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.520
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     20.04
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           1.27e-06
Time:                        21:12:13   Log-Likelihood:                -131.11
No. Observations:                  40   AIC:                             268.2
Df Residuals:                      37   BIC:                             273.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.2622      6.000      3.210      0.0

*Your answer:*

Together, income and age explain about 52% of the variation in spending (R^2 = 0.52, adjusted R^2 = 0.49). Income is a significant predictor: each extra $1k of income goes with about $0.36k more spending, holding age constant. Age is not signifcant once income is in the model.



### Q4. Which region has the most customers, and what is the average income overall?

In [12]:
# hint: count a categorical column
# take the mean of a numeric column

print(df["region"].value_counts())
print(df["income"].mean())

region
East     21
West     10
South     9
Name: count, dtype: int64
86.46


*Your answer:*
East has the most customers, and the average income is about 86.46 (roughly $86,460).


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:*
The data types themselves decided it. Two continous variabbles is correlation. Two categorical variables is chi square. A continous outcome with multiple predictors is multiple regression. Counting or averaging a single variable is descriptive statistics.

**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:*

No. Correlation only shows that income and spending move together. A third factor could drive both, or spending could even affect income. Only a controlled experiment could show causation.

## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [13]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?

exp_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)
print(exp_df)
print((expected >= 5).all())

subscribed    No   Yes
region                
East        10.5  10.5
South        4.5   4.5
West         5.0   5.0
False


*Your answer:*




### Push your code to GitHub